<a href="https://colab.research.google.com/github/owolfson/moon-dev-ai-agents/blob/main/Chatterbox_TTS_Turbo_Documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
%%bash
set -e

cd /content

# Download micromamba into /content/bin/micromamba
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create a clean env (isolated from Colab’s global packages)
./bin/micromamba create -y -n cb311 -c conda-forge python=3.11 pip

echo "✅ micromamba ready at /content/bin/micromamba"
echo "✅ env created: cb311"


In [2]:
%%capture
%%bash
set -euo pipefail

cd /content
MICROMAMBA="/content/bin/micromamba"

ts() { date +"[%Y-%m-%d %H:%M:%S]"; }

echo "$(ts) Sanity check micromamba:"
ls -lah "$MICROMAMBA"

echo "$(ts) Upgrading pip tooling inside cb311..."
"$MICROMAMBA" run -n cb311 python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CUDA 12.1)... (this can take a while)"
"$MICROMAMBA" run -n cb311 pip install \
  --progress-bar on \
  torch==2.5.1+cu121 torchaudio==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

echo "$(ts) Installing ONNX (wheel)..."
"$MICROMAMBA" run -n cb311 pip install --progress-bar on onnx==1.16.0

echo "$(ts) Installing Chatterbox package (from GitHub, no-cache, upgrade)..."
"$MICROMAMBA" run -n cb311 pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n cb311 pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  "chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master"

echo "$(ts) ✅ Installation complete!"


In [3]:
%%capture
%%bash
set -e

/content/bin/micromamba run -n cb311 python - <<'PY'
import inspect, torch
import chatterbox.tts_turbo as t

print("✅ torch:", torch.__version__)
print("✅ cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ gpu:", torch.cuda.get_device_name(0))

print("✅ chatterbox.tts_turbo path:", t.__file__)

src = inspect.getsource(t.ChatterboxTurboTTS.from_pretrained)
print("\n--- from_pretrained() (first ~80 lines) ---")
print("\n".join(src.splitlines()[:80]))

# Heuristic check for the common buggy pattern that forces token=True semantics
markers = [" or True", "token=True", "token = True", "use_auth_token=True"]
hits = [m for m in markers if m in src]
print("\nHeuristic auth-forcing markers found:", hits)

if hits:
    raise SystemExit(
        "\n❌ This install still appears to force HF auth.\n"
        "Re-run Cell 2 (it already uses --no-cache-dir --upgrade).\n"
    )

print("\n✅ Looks good: Turbo should download without requiring user tokens.")
PY

In [4]:
%%capture
%%bash
set -e

echo "Patching Chatterbox to handle missing watermarker..."

/content/bin/micromamba run -n cb311 python - <<'PY'
import sys
from pathlib import Path

# Find the tts_turbo.py file
import chatterbox.tts_turbo as t
turbo_file = Path(t.__file__)

print(f"Patching: {turbo_file}")

# Read the file
content = turbo_file.read_text()

# Patch 1: Fix watermarker initialization
old_init = "        self.watermarker = perth.PerthImplicitWatermarker()"
new_init = """        try:
            self.watermarker = perth.PerthImplicitWatermarker() if perth.PerthImplicitWatermarker is not None else None
        except (TypeError, AttributeError):
            print("Warning: Watermarker not available, skipping...")
            self.watermarker = None"""

# Patch 2: Fix watermark application in generate method
old_watermark = "        watermarked_wav = self.watermarker.apply_watermark(wav, sample_rate=self.sr)"
new_watermark = """        if self.watermarker is not None:
            watermarked_wav = self.watermarker.apply_watermark(wav, sample_rate=self.sr)
        else:
            watermarked_wav = wav"""

patches_applied = 0

if old_init in content:
    content = content.replace(old_init, new_init)
    print("✅ Patched watermarker initialization")
    patches_applied += 1
else:
    print("⚠️  Init pattern not found or already patched")

if old_watermark in content:
    content = content.replace(old_watermark, new_watermark)
    print("✅ Patched watermark application in generate()")
    patches_applied += 1
else:
    print("⚠️  Watermark apply pattern not found or already patched")

if patches_applied > 0:
    turbo_file.write_text(content)
    print(f"\n✅ Applied {patches_applied} patch(es) successfully")
else:
    print("\n⚠️  No patches applied - file may already be patched or structure changed")

PY

echo "✅ Patch complete!"

In [ ]:
# @title 4. Install Server + Run With Full Live Logs (foreground)
import os, time, subprocess, socket, requests
from pathlib import Path

PORT = 8004
REPO_DIR = "/content/Chatterbox-TTS-Server"
LOG_STDOUT = "/content/chatterbox_server_stdout.log"

def sh(cmd, check=False):
    return subprocess.run(["bash", "-lc", cmd], check=check)

def port_open(host="127.0.0.1", port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

os.chdir("/content")

# Fresh clone
sh("rm -rf /content/Chatterbox-TTS-Server", check=False)
sh("git clone https://github.com/devnen/Chatterbox-TTS-Server.git", check=True)
os.chdir(REPO_DIR)

print("=== Quick system checks ===")
sh("nvidia-smi || true", check=False)

print("\n=== Installing server requirements (prefer repo pins if present) ===")
if Path("requirements-nvidia.txt").exists():
    sh("/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel", check=False)
    sh("/content/bin/micromamba run -n cb311 pip install -r requirements-nvidia.txt", check=False)
else:
    sh(
        "/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel && "
        "/content/bin/micromamba run -n cb311 pip install "
        "fastapi 'uvicorn[standard]' pyyaml soundfile librosa safetensors "
        "python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm "
        "pydub audiotsm praat-parselmouth",
        check=False
    )

print("\n=== Removing old stdout log ===")
Path(LOG_STDOUT).unlink(missing_ok=True)

print("\n=== Starting server with LIVE logs ===")
print("Log file:", LOG_STDOUT)
print("To stop the server, run Cell 5.\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Put HF cache somewhere inspectable/persistent for this runtime
env["HF_HOME"] = "/content/hf_home"
env["TRANSFORMERS_CACHE"] = "/content/hf_home/transformers"
env["HF_HUB_CACHE"] = "/content/hf_home/hub"
Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    ["/content/bin/micromamba", "run", "-n", "cb311", "python", "-u", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

with open(LOG_STDOUT, "w", encoding="utf-8", errors="replace") as f:
    shown_link = False
    while True:
        line = proc.stdout.readline()
        if line:
            print(line, end="")
            f.write(line)
            f.flush()

        if (not shown_link) and port_open():
            shown_link = True
            print("\n=== Server port is reachable ===")
            print("Click the Colab proxy link below to open the Web UI.")
            from google.colab.output import serve_kernel_port_as_window
            serve_kernel_port_as_window(PORT)

            # Verify model load status via server endpoint
            try:
                mi = requests.get(f"http://127.0.0.1:{PORT}/api/model-info", timeout=2).json()
                print("\n/api/model-info:", mi)
            except Exception as e:
                print("\n/api/model-info query failed:", repr(e))

        if proc.poll() is not None:
            print("\n=== Server process exited with code", proc.returncode, "===")
            break

=== Quick system checks ===

=== Installing server requirements (prefer repo pins if present) ===

=== Removing old stdout log ===

=== Starting server with LIVE logs ===
Log file: /content/chatterbox_server_stdout.log
To stop the server, run Cell 5.

/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Parselmouth library not found. Unvoiced segment removal feature will be disabled.
2026-03-18 04:01:20 [INFO] __main__: Starting TTS Server directly on http://0.0.0.0:8004
2026-03-18 04:01:20 [INFO] __main__: API documentation will be available at http://0.0.0.0:8004/docs
2026-03-18 04:01:20 [INFO] __main__: Web UI will be available at http://0.0.0.0:8004/
INFO:     Started server process [9359]
INFO:     Waiting for application startup.
2026-03-18 04:01:21 [INFO] server: TTS Server: Initializing application.

<IPython.core.display.Javascript object>


/api/model-info: {'loaded': True, 'type': 'turbo', 'class_name': 'ChatterboxTurboTTS', 'device': 'cuda', 'sample_rate': 24000, 'supports_paralinguistic_tags': True, 'available_paralinguistic_tags': ['laugh', 'chuckle', 'sigh', 'gasp', 'cough', 'clear throat', 'sniff', 'groan', 'shush'], 'turbo_available_in_package': True, 'multilingual_available_in_package': True, 'supports_multilingual': False, 'supported_languages': {'en': 'English'}}
2026-03-18 04:02:23 [INFO] server: Attempting to open web browser to: http://localhost:8004/
2026-03-18 04:05:44 [INFO] server: Request received for main UI page ('/').
2026-03-18 04:05:45 [INFO] server: Request received for /api/ui/initial-data.
2026-03-18 04:05:45 [INFO] utils: Found 28 predefined voices in /content/Chatterbox-TTS-Server/voices
2026-03-18 04:05:49 [INFO] server: Request received for /save_settings.
2026-03-18 04:05:49 [INFO] config: TTS processing device resolved to: cuda
2026-03-18 04:05:49 [INFO] config: Configuration successfully 